In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
from tabpfn import TabPFNClassifier
from sklearn.model_selection import train_test_split
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import AllChem, DataStructs
import numpy as np
import torch

In [2]:
train = pd.read_csv('/home/ychen3338/proj_2/melting/m_train.csv')
test = pd.read_csv('/home/ychen3338/proj_2/melting/m_test.csv')

In [3]:
class morgan_fp:
    def __init__(self, radius, length):
        self.radius = radius
        self.length = length
    def __call__(self, smiles):
        mol = Chem.MolFromSmiles(smiles)
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, self.length)
        npfp = np.array(list(fp.ToBitString())).astype('float32')
        return npfp

In [4]:
def conv_data(data, fp):
    data['c-fp'] = data['Cation'].apply(fp)
    x_c=np.array(list(data['c-fp']))
    data['a-fp'] = data['Anion'].apply(fp)
    x_a=np.array(list(data['a-fp']))
    xx = np.concatenate([x_c, x_a], axis =1)
    y = data['Tm'].values
    return xx, y

In [5]:
params ={'bagging_temperature': 134.9836915147014, 'depth': 6, 
         'fp_length': 1361, 'fp_radius': 3, 
         'iterations': 746, 'l2_leaf_reg': 3.026090580066783, 
         'learning_rate': 0.02471776939967432, 
         'random_strength': 1.3848955442798498}

In [5]:
result = pd.read_csv('/home/ychen3338/project_2/code/melting/melting_MF.csv')
result.sort_values('loss', ascending= True, inplace = True)
result.reset_index(drop = True, inplace =True)
import ast
params = result.loc[0, 'params']
params = ast.literal_eval(params)

In [6]:
params

{'bagging_temperature': 176.86662206316092,
 'depth': 6,
 'fp_length': 266,
 'fp_radius': 3,
 'iterations': 910,
 'l2_leaf_reg': 3.03715573091437,
 'learning_rate': 0.02496159641721271,
 'random_strength': 1.0718610593918538}

In [8]:
fp = morgan_fp(params['fp_radius'], params['fp_length'])

In [9]:
X_train, y_train = conv_data(train, fp)
X_test, y_test = conv_data(test, fp)


[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerator
[14:17:24] DEPRECATION WARNING: please use MorganGenerat

In [10]:
num_bins = 8
all_y = np.concatenate([y_train, y_test])
bins = np.linspace(all_y.min(), all_y.max(), num_bins + 1)

y_train_bin = np.digitize(y_train, bins) - 1
y_test_bin = np.digitize(y_test, bins) - 1

all_bin = np.concatenate([y_train_bin, y_test_bin])
le = LabelEncoder()
all_encoded = le.fit_transform(all_bin)
y_train_enc = all_encoded[:len(y_train)]
y_test_enc = all_encoded[len(y_train):]

In [11]:
num_bins = 9  # 修改为跟 proba.shape[1] 一致
bins = np.linspace(all_y.min(), all_y.max(), num_bins + 1)

y_train_bin = np.digitize(y_train, bins) - 1
y_test_bin = np.digitize(y_test, bins) - 1

# 确保不超出 bin 数范围
y_train_bin[y_train_bin >= num_bins] = num_bins - 1
y_test_bin[y_test_bin >= num_bins] = num_bins - 1

# Label encoding
all_bin = np.concatenate([y_train_bin, y_test_bin])
le = LabelEncoder()
all_encoded = le.fit_transform(all_bin)
y_train_enc = all_encoded[:len(y_train)]
y_test_enc = all_encoded[len(y_train):]

# bin centers 正确构建
bin_centers = 0.5 * (bins[:-1] + bins[1:])


In [12]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
model = TabPFNClassifier(device=device, ignore_pretraining_limits=True)
model.fit(X_train, y_train_enc)

Using device: cuda


/home/ychen3338/miniconda3/envs/ml/lib/python3.10/site-packages/tabpfn/classifier.py:422: UserWarning: Number of features 532 is greater than the maximum Number of features 500 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(


TabPFNClassifier(device='cuda', ignore_pretraining_limits=True)

In [13]:
proba = model.predict_proba(X_test)

bin_centers = 0.5 * (bins[:-1] + bins[1:])

regression_preds = np.dot(proba, bin_centers)


In [14]:
r2 = r2_score(y_test, regression_preds)
rmse = mean_squared_error(y_test, regression_preds, squared=False)
print(f"R² score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

R² score: 0.7533
RMSE: 30.7780


/home/ychen3338/miniconda3/envs/ml/lib/python3.10/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
